In [22]:
import os
import tarfile
import hashlib
import json
import logging
from tqdm import tqdm
from datetime import datetime


class SafeCompressor:
    def __init__(self, list_folder_ignore=None, log_dir="logs"):
        self.list_folder_ignore = list_folder_ignore or []
        self.logger = self._setup_logger(log_dir)

    def _setup_logger(self, log_dir):
        os.makedirs(log_dir, exist_ok=True)
        log_name = datetime.now().strftime("compressor_%Y-%m-%d_%H-%M-%S.log")
        log_path = os.path.join(log_dir, log_name)

        logger = logging.getLogger(log_name)
        logger.setLevel(logging.DEBUG)
        formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')

        # Arquivo
        file_handler = logging.FileHandler(log_path)
        file_handler.setFormatter(formatter)
        logger.addHandler(file_handler)

        # Console
        console_handler = logging.StreamHandler()
        console_handler.setFormatter(formatter)
        logger.addHandler(console_handler)

        logger.info("Logger inicializado.")
        return logger

    def _gerar_hash_arquivo(self, file_path):
        sha256 = hashlib.sha256()
        try:
            with open(file_path, "rb") as f:
                for bloco in iter(lambda: f.read(4096), b""):
                    sha256.update(bloco)
        except Exception as e:
            self.logger.error(f"Erro ao gerar hash de {file_path}: {e}")
            return None
        return sha256.hexdigest()

    def compactar(self, path_origin, path_destino):
        nome_base = os.path.basename(path_origin.rstrip("/"))
        compressed_path = os.path.join(path_destino, nome_base + ".tar.gz")
        hash_path = os.path.join(path_destino, nome_base + "_hashes.json")

        self.logger.info(f"Iniciando compactação de: {path_origin}")
        arquivos = []

        for root, dirs, files in os.walk(path_origin):
            dirs[:] = [d for d in dirs if d not in self.list_folder_ignore]
            for file in files:
                arquivos.append(os.path.join(root, file))

        hashes = {}
        try:
            with tarfile.open(compressed_path, "w:gz") as tar:
                for arquivo in tqdm(arquivos, desc="Compactando arquivos", unit="arquivo"):
                    rel_path = os.path.relpath(arquivo, start=os.path.dirname(path_origin))
                    tar.add(arquivo, arcname=rel_path)
                    hash_value = self._gerar_hash_arquivo(arquivo)
                    if hash_value:
                        hashes[rel_path] = hash_value

            with open(hash_path, "w") as f:
                json.dump(hashes, f, indent=4)

            self.logger.info(f"✅ Compactação finalizada: {compressed_path}")
            self.logger.info(f"✅ Hashes salvos em: {hash_path}")
        except Exception as e:
            self.logger.error(f"❌ Erro durante a compactação: {e}")

    def descompactar(self, tar_path, output_dir):
        self.logger.info(f"Iniciando descompactação de: {tar_path}")
        try:
            os.makedirs(output_dir, exist_ok=True)
            with tarfile.open(tar_path, "r:gz") as tar:
                tar.extractall(path=output_dir)
            self.logger.info(f"✅ Descompactado com sucesso em: {output_dir}")
        except Exception as e:
            self.logger.error(f"❌ Erro ao descompactar: {e}")

    def verificar_integridade(self, pasta_extraida, hash_file_path):
        self.logger.info("Iniciando verificação de integridade.")
        try:
            with open(hash_file_path, "r") as f:
                hashes_salvos = json.load(f)

            erros = []
            for rel_path, hash_original in tqdm(hashes_salvos.items(), desc="Verificando integridade"):
                caminho_absoluto = os.path.join(pasta_extraida, rel_path)
                if not os.path.exists(caminho_absoluto):
                    msg = f"Arquivo ausente: {rel_path}"
                    erros.append(msg)
                    self.logger.warning(msg)
                    continue

                hash_atual = self._gerar_hash_arquivo(caminho_absoluto)
                if not hash_atual or hash_atual != hash_original:
                    msg = f"Hash diferente: {rel_path}"
                    erros.append(msg)
                    self.logger.warning(msg)

            if not erros:
                self.logger.info("✅ Todos os arquivos verificados com sucesso!")
            else:
                self.logger.warning(f"⚠️ {len(erros)} problemas encontrados na verificação de integridade.")
                log_erro_path = os.path.join(os.path.dirname(hash_file_path), "integrity_failures.log")
                with open(log_erro_path, "w") as f:
                    for erro in erros:
                        f.write(erro + "\n")
                self.logger.warning(f"⚠️ Erros salvos em: {log_erro_path}")
        except Exception as e:
            self.logger.error(f"❌ Erro na verificação de integridade: {e}")


In [23]:
compressor = SafeCompressor(list_folder_ignore=["audio", "video", "image", "recortes"])

path_origin_comp = "/media/guilherme/ssd_m2_data/py_new_projects/fast_english/database/extract_data_video/data/extracted_data"
path_destino_comp = "/media/guilherme/ssd_m2_data/py_new_projects/fast_english/app/toolkit/data"


path_origin_descomp = "/media/guilherme/ssd_m2_data/py_new_projects/fast_english/app/toolkit/data/extracted_data.tar.gz"
path_destino_descomp = "/media/guilherme/ssd_m2_data/py_new_projects/fast_english/database/extract_data_video/data"

# Compactar
compressor.compactar(
    path_origin=path_origin_comp,
    path_destino=path_destino_comp
)

# Descompactar
compressor.descompactar(
    tar_path=path_origin_descomp,
    output_dir=path_destino_descomp
)

# Verificar integridade
compressor.verificar_integridade(
    pasta_extraida=path_destino_descomp,
    hash_file_path=os.path.join(path_destino_comp, "extracted_data_hashes.json")
)


2025-05-19 06:18:27,385 - INFO - Logger inicializado.
2025-05-19 06:18:27,386 - INFO - Iniciando compactação de: /media/guilherme/ssd_m2_data/py_new_projects/fast_english/database/extract_data_video/data/extracted_data
Compactando arquivos: 100%|██████████| 43308/43308 [00:43<00:00, 1006.02arquivo/s]
2025-05-19 06:19:10,899 - INFO - ✅ Compactação finalizada: /media/guilherme/ssd_m2_data/py_new_projects/fast_english/app/toolkit/data/extracted_data.tar.gz
2025-05-19 06:19:10,900 - INFO - ✅ Hashes salvos em: /media/guilherme/ssd_m2_data/py_new_projects/fast_english/app/toolkit/data/extracted_data_hashes.json
2025-05-19 06:19:10,902 - INFO - Iniciando descompactação de: /media/guilherme/ssd_m2_data/py_new_projects/fast_english/app/toolkit/data/extracted_data.tar.gz
2025-05-19 06:19:28,921 - INFO - ✅ Descompactado com sucesso em: /media/guilherme/ssd_m2_data/py_new_projects/fast_english/database/extract_data_video/data
2025-05-19 06:19:28,950 - INFO - Iniciando verificação de integridade.
V

In [13]:
os.path.exists("/media/guilherme/ssd_m2_data/py_new_projects/fast_english/database/extract_data_video/extracted_data/phrases/data_organize/viagem/no_posto_de_turismo/where's_the_cathedral")

False

In [14]:
os.path.dirname(pasta_extraida)

'/media/guilherme/ssd_m2_data/py_new_projects/fast_english/database/extract_data_video'

In [12]:
caminho_absoluto

"/media/guilherme/ssd_m2_data/py_new_projects/fast_english/database/extract_data_video/extracted_data/phrases/data_organize/viagem/no_posto_de_turismo/where's_the_cathedral/text_v2.json"

In [ ]:
# /media/guilherme/ssd_m2_data/py_new_projects/fast_english/database/extract_data_video/data/extracted_data/phrases/data_organize/viagem/no_posto_de_turismo/where's_the_cathedral/text_v2.json
# /media/guilherme/ssd_m2_data/py_new_projects/fast_english/database/extract_data_video/extracted_data/phrases/data_organize/viagem/no_posto_de_turismo/where's_the_cathedral/text_v2.json

In [1]:
from tqdm import tqdm
import tarfile
import os

def compactar_pasta_com_caminhos_fixos(path_origin, path_destivo, list_folder_ignore=None):
    """
    Compacta uma pasta em um arquivo .tar.gz, ignorando pastas especificadas.

    Args:
        path_origin (str): Caminho da pasta a ser compactada.
        list_folder_ignore (list): Lista de nomes de pastas a serem ignoradas.
    """
    # Nome do arquivo compactado
    compressed_path = os.path.join(path_destivo, os.path.basename(path_origin) + ".tar.gz")

    try:
        # Obter a lista de arquivos para calcular o progresso
        arquivos = []
        for root_dir, dirs, files in os.walk(path_origin):
            dirs[:] = [d for d in dirs if d not in (list_folder_ignore or [])]

            for file in files:
                arquivos.append(os.path.join(root_dir, file))

        # Compactação usando tarfile com barra de progresso
        with tarfile.open(compressed_path, "w:gz") as tar:
            for arquivo in tqdm(arquivos, desc="Compactando arquivos", unit="arquivo"):
                tar.add(arquivo, arcname=os.path.relpath(arquivo, start=os.path.dirname(path_origin)))

        print(f"Pasta compactada com sucesso: {compressed_path}")
    except Exception as e:
        print(f"Erro ao compactar a pasta: {e}")

# Lista de pastas a serem ignoradas
list_folder_ignore = ["audio", "video", "image", "recortes"]


# Exemplo de uso
path_origin = "/media/guilherme/ssd_m2_data/py_new_projects/fast_english/database/extract_data_video/data/extracted_data"
path_destivo = "/media/guilherme/ssd_m2_data/py_new_projects/fast_english/app/toolkit/data"
compactar_pasta_com_caminhos_fixos(path_origin, path_destivo, list_folder_ignore)

Compactando arquivos: 100%|██████████| 43308/43308 [01:25<00:00, 508.02arquivo/s]


Pasta compactada com sucesso: /media/guilherme/ssd_m2_data/py_new_projects/fast_english/app/toolkit/data/extracted_data.tar.gz


In [ ]:
dedede = r"C:\pyprojects\fast-english\database\extract_data_video\data\extracted_data"

destino = r"C:\pyprojects\fast-english\app\toolkit\data"

compactar_pasta_com_caminhos_fixos(dedede, destino, list_folder_ignore)


Compactando arquivos:  69%|██████▉   | 29849/43308 [04:48<02:04, 108.33arquivo/s]

In [2]:
import tarfile
import os

def descompactar_tar_gz(tar_gz_path, output_dir):
    """Descompacta um arquivo .tar.gz para o diretório de saída especificado."""
    if not tar_gz_path.endswith(".tar.gz"):
        print(f"Erro: O arquivo {tar_gz_path} não é um arquivo .tar.gz válido.")
        return None

    try:
        # Certifique-se de que o diretório de saída existe
        os.makedirs(output_dir, exist_ok=True)
        print(f"Diretório de saída garantido: {output_dir}")

        # Extrair o arquivo .tar.gz
        with tarfile.open(tar_gz_path, "r:gz") as tar:
            print(f"Extraindo arquivos de {tar_gz_path} para {output_dir}...")
            tar.extractall(path=output_dir)

        print(f"Arquivo descompactado com sucesso em: {output_dir}")
        return output_dir
    except FileNotFoundError:
        print(f"Erro: O arquivo {tar_gz_path} não foi encontrado.")
    except PermissionError:
        print(f"Erro: Permissão negada ao acessar {tar_gz_path} ou {output_dir}.")
    except Exception as e:
        print(f"Erro inesperado ao descompactar o arquivo {tar_gz_path}: {e}")

    return None

# Exemplo de uso
path_origin = "/media/guilherme/ssd_m2_data/py_new_projects/fast_english/app/toolkit/data/extracted_data.tar.gz"
output_directory = "/media/guilherme/ssd_m2_data/py_new_projects/fast_english/database/extract_data_video/data"

descompactar_tar_gz(path_origin, output_directory)

Diretório de saída garantido: /media/guilherme/ssd_m2_data/py_new_projects/fast_english/database/extract_data_video/data
Extraindo arquivos de /media/guilherme/ssd_m2_data/py_new_projects/fast_english/app/toolkit/data/extracted_data.tar.gz para /media/guilherme/ssd_m2_data/py_new_projects/fast_english/database/extract_data_video/data...
Arquivo descompactado com sucesso em: /media/guilherme/ssd_m2_data/py_new_projects/fast_english/database/extract_data_video/data


'/media/guilherme/ssd_m2_data/py_new_projects/fast_english/database/extract_data_video/data'

In [ ]:
import gzip
import shutil

def descompactar_fasta_interativo():
    # Inicializar a interface Tkinter
    root = tk.Tk()
    root.withdraw()  # Ocultar a janela principal do Tkinter

    # Selecionar o arquivo .gz de origem
    gz_path = filedialog.askopenfilename(title="Selecione o arquivo .gz para descompactar", filetypes=[("Arquivos GZ", "*.gz")])

    if not gz_path:
        print("Nenhum arquivo selecionado.")
        return

    if not gz_path.endswith(".gz"):
        print("O arquivo selecionado não é um arquivo .gz")
        return

    # Selecionar o local de destino
    output_dir = filedialog.askdirectory(title="Selecione o diretório de destino")

    if not output_dir:
        print("Nenhum diretório de destino selecionado.")
        return

    # Caminho do arquivo descompactado
    output_path = os.path.join(output_dir, os.path.basename(gz_path)[:-3])  # Remove o .gz

    try:
        # Descompactar o arquivo
        with gzip.open(gz_path, 'rb') as f_in:
            with open(output_path, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)

        print(f"Arquivo descompactado com sucesso: {output_path}")
    except Exception as e:
        print(f"Erro ao descompactar o arquivo: {e}")

# Exemplo de uso
descompactar_fasta_interativo()